# Epithelium segmenter training

Trains a U-Net (ResNet-34 encoder) to find epithelium in whole images, using the 50 MedSAM-assisted masks in `annotations/epithelium/masks/` (in the repo). Patient-level 5-fold cross-validation: every image is tested once by a model that never saw its patient. Baselines on the same images: *tissue* (all tissue counts as epithelium) and *whole* (the whole image).

Code: [oral-cancer-detection](https://github.com/Duggineniakhil/oral-cancer-detection) · `scripts/segmentation/train_segmenter.py`

### Before running
- **Add Input** → the dataset made from `kaggle_upload/epithelium-annotation-images.zip` (the same one used for the embeddings).
- **Settings** → Accelerator **GPU T4**, Internet **On**.
- **Run All** (about 20–30 min), then download `segmenter.zip` with the link at the end.

## 1. Setup

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU! Turn it on in Settings -> Accelerator before continuing.")

In [ ]:
try:
    import segmentation_models_pytorch
except ImportError:
    !pip install -q segmentation-models-pytorch

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Duggineniakhil/oral-cancer-detection.git"
CODE_DIR = "/tmp/oral-cancer-detection"

if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only"], check=True)
print(subprocess.run(["git", "-C", CODE_DIR, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

matches = list(Path("/kaggle/input").rglob("annotations/epithelium/annotation_set.csv"))
assert matches, "annotation_set.csv not found under /kaggle/input. Did you add the dataset as an input?"

DATA_ROOT = matches[0].parent.parent.parent
OUTPUT_ROOT = Path("/kaggle/working")
os.environ["ORAL_CANCER_DATA_ROOT"] = str(DATA_ROOT)
os.environ["ORAL_CANCER_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

print("Data root:  ", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Masks in repo:", len(list(Path(CODE_DIR, "annotations/epithelium/masks").glob("*.png"))))

## 2. Train (5 rounds)

In [ ]:
!cd {CODE_DIR} && python -m scripts.segmentation.train_segmenter

## 3. Results

In [ ]:
import json
import pandas as pd
from IPython.display import Image, display

RUN = OUTPUT_ROOT / "runs" / "segmenter"
print(json.dumps(json.loads((RUN / "summary.json").read_text()), indent=2))
display(pd.read_csv(RUN / "test_per_image.csv")[["image_path", "zoom", "segmenter_dice", "tissue_dice", "whole_dice"]].round(3))
display(Image(str(RUN / "examples.png")))

## 4. Download

Unzip `segmenter.zip` into the project folder on your laptop. It creates `runs/segmenter/`.

In [ ]:
import shutil
from IPython.display import FileLink

archive = shutil.make_archive(str(OUTPUT_ROOT / "segmenter"), "zip", OUTPUT_ROOT, "runs/segmenter")
print(archive, round(os.path.getsize(archive) / 2**20, 1), "MB")
FileLink("segmenter.zip")